# 05b — Pooled Dataset Assembly (Bogor + Warsaw)

Runs AFTER each city's own `05_dataset_assembly.ipynb` (or can be run
standalone — it re-does the same scan/audit per city itself, so it
doesn't actually depend on `05` having been run first). Requires `04b`
(vocab unification) to have already been run and swapped in, or the
pooled model's shared embedding tables will be silently misaligned.

Produces ONE `dataset_index.parquet` with a `city` column plus explicit
`svg_dir`/`tvg_dir` path columns per row, so `07`'s data loader never has
to re-derive a city's folders from just the city name.

**Fold semantics for pooling:** `fold_rep{r}` was assigned independently
per city (separate spatial KMeans, since Bogor/Warsaw sit in different
UTM zones and aren't spatially comparable). Both cities used the SAME
`configs/eval.yaml` (`k_folds`, `repeats`), so `fold_rep{r} == i` simply
means "partition i" within each city — pooling reuses these integers
as-is: pooled fold i = (Bogor fold i) ∪ (Warsaw fold i). This keeps
each city's own spatial blocking intact while ensuring every pooled
fold contains both cities. Per-fold normalization in `07` then fits on
the combined (both-city) training partition of each fold — same
"fit on train partition only" rule as before, just extended to both
cities' rows.

**Not built here:** cross-city generalization splits (train-all-X /
test-all-Y) — that's a separate track handled directly in `07`, since
it isn't a fold scheme at all.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric pyyaml pandas tqdm

In [ ]:
from pathlib import Path

BOGOR_BASE_DIR  = Path("/content/drive/MyDrive/crash-dualgraph/data/bogor")
WARSAW_BASE_DIR = Path("/content/drive/MyDrive/crash-dualgraph/data/warsaw")
COMBINED_BASE_DIR = Path("/content/drive/MyDrive/crash-dualgraph/data/combined")

CITY_DIRS = {
    "bogor":  {"interim": BOGOR_BASE_DIR / "interim",  "processed": BOGOR_BASE_DIR / "processed"},
    "warsaw": {"interim": WARSAW_BASE_DIR / "interim", "processed": WARSAW_BASE_DIR / "processed"},
}

COMBINED_PROCESSED_DIR = COMBINED_BASE_DIR / "processed"
COMBINED_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INDEX_OUT = COMBINED_PROCESSED_DIR / "dataset_index.parquet"

In [ ]:
import dataset_audit
import torch
import pandas as pd
from tqdm.auto import tqdm

per_city_final = {}
per_city_stats = {}

for city, d in CITY_DIRS.items():
    interim_dir, processed_dir = d["interim"], d["processed"]
    svg_dir, tvg_dir = processed_dir / "svg_graphs", processed_dir / "tvg_graphs"

    reconciled = pd.read_parquet(interim_dir / "reconciled_points.parquet")
    all_point_ids = reconciled["point_id"].tolist()

    status_df, node_stats, edge_stats = dataset_audit.scan_dataset(
        tqdm(all_point_ids, desc=f"[{city}] scanning SVG+TVG"), svg_dir, tvg_dir, torch
    )
    report = dataset_audit.build_drop_report(status_df)
    final_df = dataset_audit.filter_complete_points(reconciled, status_df)

    final_df = final_df.copy()
    final_df["city"] = city
    final_df["svg_dir"] = str(svg_dir)
    final_df["tvg_dir"] = str(tvg_dir)
    # point_id stays UNCHANGED here — it's still the on-disk filename
    # ({point_id}.pt inside this city's svg_dir/tvg_dir) and must keep
    # matching that exactly. The globally-unique key is built separately
    # below, once both cities are known, as its own 'uid' column.

    print(f"[{city}] total={report['total_points']}  complete={report['complete_pairs']}  "
          f"dropped={report['total_points'] - report['complete_pairs']}")
    if report["missing_svg_only"] or report["missing_tvg_only"] or report["missing_both"]:
        print(f"    missing_svg_only={report['missing_svg_only']}  "
              f"missing_tvg_only={report['missing_tvg_only']}  missing_both={report['missing_both']}")

    per_city_final[city] = final_df
    per_city_stats[city] = (node_stats, edge_stats, len(final_df))

In [ ]:
# ── Point-id collision check + globally-unique 'uid' ──────────────
# point_id is only guaranteed unique WITHIN a city's own graph folders
# (it's the on-disk {point_id}.pt filename, left untouched above for
# that reason). Rather than just flag a possible collision, always
# build an explicit 'uid' as the real primary key for the pooled index —
# removes any chance of a later naive merge/groupby on bare point_id
# silently conflating rows from different cities, collision or not.
#
# uid FORMAT: "{city_prefix}_{positive|negative}_{n}", e.g. "bog_positive_12",
# "war_negative_34" — encodes BOTH city and label directly in the key
# (not just city), and n is a per-(city, label) running index (0, 1, 2, ...)
# rather than the raw point_id. This makes uid self-describing on sight
# (no join needed to know a row's city or label) at the cost of n no
# longer matching point_id numerically — point_id remains the on-disk
# {point_id}.pt filename (untouched, see cell above); uid is purely a
# pooled-index primary key, never used to resolve a file path.
CITY_PREFIX = {"bogor": "bog", "warsaw": "war"}
LABEL_NAME = {1: "positive", 0: "negative"}

bogor_ids = set(per_city_final["bogor"]["point_id"])
warsaw_ids = set(per_city_final["warsaw"]["point_id"])
overlap = bogor_ids & warsaw_ids
if overlap:
    print(f"⚠️  {len(overlap)} point_id values appear in BOTH cities: {list(overlap)[:10]}...")
else:
    print("✅ No raw point_id collisions between cities (uid encodes city+label regardless).")

for city, prefix in CITY_PREFIX.items():
    df = per_city_final[city].copy()
    # Running index computed PER (city, label) group, in the row's
    # current order — stable and reproducible given final_df's own
    # ordering upstream, but NOT meant to be re-derived from point_id;
    # treat uid as an opaque key once assigned.
    df["_label_name"] = df["label"].map(LABEL_NAME)
    df["_running_idx"] = df.groupby("_label_name").cumcount()
    df["uid"] = prefix + "_" + df["_label_name"] + "_" + df["_running_idx"].astype(str)
    df = df.drop(columns=["_label_name", "_running_idx"])
    per_city_final[city] = df

assert per_city_final["bogor"]["uid"].is_unique and per_city_final["warsaw"]["uid"].is_unique
combined_uid_check = pd.concat([per_city_final["bogor"]["uid"], per_city_final["warsaw"]["uid"]])
assert combined_uid_check.is_unique, "uid collided across cities — should be impossible given the prefix, investigate."
print("'uid' column added (e.g. bog_positive_12, war_negative_34) — use this as the pooled dataset's primary key.")


In [ ]:
# ── Concatenate into the pooled index ───────────────────────────
pooled_df = pd.concat(per_city_final.values(), ignore_index=True)

fold_cols = [c for c in pooled_df.columns if c.startswith("fold_rep")]
print(f"Pooled dataset: {len(pooled_df)} rows ({len(per_city_final['bogor'])} bogor + "
      f"{len(per_city_final['warsaw'])} warsaw)")
print(f"Fold columns: {fold_cols}")
print(f"\nOverall class balance: {(pooled_df['label']==1).sum()} positive, "
      f"{(pooled_df['label']==0).sum()} negative")
print(f"\nPer-city breakdown:")
display(pooled_df.groupby("city")["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── Pooled-fold class balance ── confirms every pooled fold actually
# contains BOTH cities (it should, by construction, but verify rather
# than assume — same discipline as the original per-city QC in 05) ────
for col in fold_cols:
    print(f"\n--- pooled {col} — by city ---")
    display(pooled_df.groupby([col, "city"])["label"].agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── Node/edge-type coverage, combined across both cities ────────────
combined_node_stats, combined_edge_stats = {}, {}
for city in per_city_stats:
    node_stats, edge_stats, _ = per_city_stats[city]
    for nt, s in node_stats.items():
        acc = combined_node_stats.setdefault(nt, {"total_count": 0, "n_graphs_present": 0})
        acc["total_count"] += s["total_count"]
        acc["n_graphs_present"] += s["n_graphs_present"]
    for ek, s in edge_stats.items():
        acc = combined_edge_stats.setdefault(ek, {"total_count": 0, "n_graphs_present": 0})
        acc["total_count"] += s["total_count"]
        acc["n_graphs_present"] += s["n_graphs_present"]

print("Node type coverage (combined):")
for nt, stats in sorted(combined_node_stats.items()):
    pct = 100 * stats["n_graphs_present"] / len(pooled_df) if len(pooled_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {nt:20s} total={stats['total_count']:6d}  present_in={stats['n_graphs_present']:4d}/{len(pooled_df)} ({pct:.1f}%){flag}")

print("\nEdge type coverage (combined):")
for ek, stats in sorted(combined_edge_stats.items(), key=lambda kv: str(kv[0])):
    pct = 100 * stats["n_graphs_present"] / len(pooled_df) if len(pooled_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {str(ek):45s} total={stats['total_count']:6d}  present_in={stats['n_graphs_present']:4d}/{len(pooled_df)} ({pct:.1f}%){flag}")

In [ ]:
# ── Sanity check: confirm highway_type_idx / building type_idx ranges
# are consistent with a UNIFIED vocab (i.e. 04b was actually run and
# swapped in) — catches the mistake of pooling pre-unification graphs
# before it corrupts a training run.
import random

sample_hw_idx = []
for city, d in CITY_DIRS.items():
    tvg_dir = d["processed"] / "tvg_graphs"
    sample_ids = random.sample(per_city_final[city]["point_id"].tolist(),
                                min(20, len(per_city_final[city])))
    for pid in sample_ids:
        g = torch.load(tvg_dir / f"{pid}.pt", weights_only=False)
        sample_hw_idx.append((city, int(g["incident"].highway_type_idx.item())))

max_idx_seen = max(i for _, i in sample_hw_idx)
print(f"Sampled highway_type_idx range across both cities: 0..{max_idx_seen}")
print("If this looks suspiciously small/city-siloed (e.g. bogor only ever shows")
print("0-12 and warsaw only ever shows 0-8, never overlapping in a way that")
print("makes sense), re-run 04b before trusting this pooled dataset.")

In [ ]:
# ── Save the pooled index ───────────────────────────────────
cols = ["uid"] + [c for c in pooled_df.columns if c != "uid"]
pooled_df = pooled_df[cols]
pooled_df.to_parquet(INDEX_OUT, index=False)
print(f"✅ Saved {len(pooled_df)} pooled rows to {INDEX_OUT}  (primary key: 'uid')")
print()
print("Next: 07 pooled-vs-separate comparison tracks (Bogor-only, Warsaw-only,")
print("pooled k-fold CV, cross-city generalization).")